In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# Example sentences
tgt_sentence = ["It", "is", "amazing", "."]
src_sentence = ["ChatGPT", "explains", "things", "clearly", "."]

tgt_len = len(tgt_sentence)
src_len = len(src_sentence)
vocab_size = 10000
embedding_dim = 16
num_heads = 2
hidden_dim = 32

In [3]:
# 1️⃣ Token embeddings
token_embedding = nn.Embedding(vocab_size, embedding_dim)
tgt_ids = torch.arange(tgt_len)
src_ids = torch.arange(src_len)
tgt_emb = token_embedding(tgt_ids)  # [tgt_len, embedding_dim]
src_emb = token_embedding(src_ids)  # [src_len, embedding_dim]

In [4]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

pos_encoder = PositionalEncoding(embedding_dim)
tgt_emb = pos_encoder(tgt_emb)  # [tgt_len, embedding_dim]
src_emb = pos_encoder(src_emb)  # [src_len, embedding_dim]

In [5]:
# 3️⃣ Tiny Transformer Decoder layer
decoder_layer = nn.TransformerDecoderLayer(
    d_model=embedding_dim,
    nhead=num_heads,
    dim_feedforward=hidden_dim
)
transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=1)

# Transformer expects [seq_len, batch_size, embedding_dim]
tgt_emb = tgt_emb.unsqueeze(1)  # [tgt_len, 1, embedding_dim]
src_emb = src_emb.unsqueeze(1)  # [src_len, 1, embedding_dim]

# Optional: causal mask for decoder (prevents attending to future tokens)
tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len)

In [6]:
# Forward pass
output = transformer_decoder(
    tgt=tgt_emb,
    memory=src_emb,
    tgt_mask=tgt_mask
)

print("Decoder output shape:", output.shape)
print("Decoder output:\n", output.squeeze(1))

Decoder output shape: torch.Size([4, 1, 16])
Decoder output:
 tensor([[ 0.4381,  0.7030,  1.7273,  0.4287, -0.2070, -0.1605, -1.8819, -0.5410,
         -0.8971,  0.0611,  0.8322,  0.1119, -0.8990,  1.6103, -1.7698,  0.4436],
        [-1.7688, -0.3755,  0.8587,  1.0956,  0.5605,  0.6165, -1.7108, -0.3754,
          0.1733,  1.1085,  0.5613,  1.8745, -0.8948, -0.8484, -0.2977, -0.5775],
        [ 0.6638, -1.4244,  2.2056,  1.0936,  0.1727, -0.1813, -0.7710, -0.6403,
         -0.4191,  0.5850, -1.0492,  0.3473,  1.6991, -0.8007, -0.8787, -0.6023],
        [ 0.2059, -0.8564,  2.7552, -1.0833,  1.0792,  1.0797, -1.3969,  0.0441,
         -0.1465, -1.0836, -0.3311,  0.3723,  0.4410, -0.6086, -0.4028, -0.0682]],
       grad_fn=<SqueezeBackward1>)
